# Identidad del dinosaurio con Ollama

A partir del nombre elegido en el generador de caracteres, Ollama (en este notebook de SageMaker)
describe la especie: apariencia, hábitat, alimentación, comportamiento y un rasgo distintivo.
Se comparan dos modelos con el mismo prompt y se conserva la identidad más coherente en `identidad.json`,
que es la entrada de la imagen y del chat.

In [ ]:
import json, time, requests

OLLAMA = "http://localhost:11434"
NOMBRE = "Aborosaurus"
MODELOS = ["llama3.2:3b", "gemma4:e2b"]
OPCIONES = {"temperature": 0.4, "seed": 42}  # baja temperatura: rasgos consistentes entre sí

disponibles = [m["name"] for m in requests.get(f"{OLLAMA}/api/tags").json()["models"]]
print("Modelos disponibles:", disponibles)
faltan = [m for m in MODELOS if m not in disponibles]
if faltan:
    print("Descargar antes en una terminal: " + "; ".join(f"docker exec ollama ollama pull {m}" for m in faltan))

## Prompt

Se le dan al modelo raíces reales de nombres de dinosaurios para que interprete el nombre y se le pide
una respuesta en JSON con campos fijos, para que la app y el generador de imágenes los usen directamente.
`prompt_imagen` va en inglés porque el modelo text-to-image funciona mejor así.

In [ ]:
SISTEMA = """Eres un paleontólogo que describe especies ficticias de dinosaurios.
Todo lo que describes debe ser coherente entre sí: el tamaño, la anatomía, el hábitat,
la dieta y el comportamiento tienen que encajar. Respondes siempre en español y solo con JSON válido."""

PLANTILLA = """Crea la identidad de un dinosaurio ficticio llamado {nombre}.

Interpreta el nombre usando raíces reales de nombres de dinosaurios, por ejemplo:
-saurus (lagarto), -raptor (ladrón), -odon (diente), -venator (cazador), -ceratops (cara con cuernos),
-mimus (imitador), -titan (gigante), arbor- (árbol), hydro- (agua), micro- (pequeño), mega- (grande).

Responde solo con un JSON con estas claves:
- "nombre": "{nombre}"
- "significado": qué significa el nombre según sus raíces (1 frase)
- "periodo": periodo geológico en el que vivió
- "apariencia": tamaño en metros, peso, postura, piel o plumas y colores (2-3 frases)
- "habitat": dónde vivía (1-2 frases)
- "alimentacion": dieta y cómo consigue el alimento, coherente con sus dientes y su tamaño (1-2 frases)
- "comportamiento": vida social, actividad y defensa (2 frases)
- "rasgo_distintivo": una característica única que lo diferencia de otros dinosaurios (1 frase)
- "prompt_imagen": descripción visual en inglés para un generador de imágenes, máximo 40 palabras,
  solo rasgos visibles y el entorno, sin el nombre"""

CLAVES = ["nombre", "significado", "periodo", "apariencia", "habitat", "alimentacion",
          "comportamiento", "rasgo_distintivo", "prompt_imagen"]
print(PLANTILLA.format(nombre=NOMBRE))

In [ ]:
def crear_identidad(modelo, nombre=NOMBRE):
    inicio = time.time()
    r = requests.post(f"{OLLAMA}/api/chat", timeout=900, json={
        "model": modelo,
        "messages": [{"role": "system", "content": SISTEMA},
                     {"role": "user", "content": PLANTILLA.format(nombre=nombre)}],
        "format": "json",
        "stream": False,
        "options": OPCIONES,
    })
    r.raise_for_status()
    identidad = json.loads(r.json()["message"]["content"])
    faltantes = [k for k in CLAVES if not identidad.get(k)]
    return identidad, round(time.time() - inicio, 1), faltantes


resultados = {}
for modelo in MODELOS:
    if modelo not in disponibles:
        continue
    identidad, segundos, faltantes = crear_identidad(modelo)
    resultados[modelo] = identidad
    with open(f"identidad_{modelo.replace(':', '_')}.json", "w", encoding="utf-8") as f:
        json.dump(identidad, f, indent=2, ensure_ascii=False)
    print(f"===== {modelo}  ({segundos} s)  claves faltantes: {faltantes or 'ninguna'}")
    for k, v in identidad.items():
        print(f"{k}: {v}")
    print()

## Elección

Revisa que cada identidad sea coherente: que el tamaño cuadre con el peso, que los dientes y la dieta
encajen, que el hábitat sea compatible con el comportamiento y que el nombre se interprete bien.
Escribe el modelo elegido y ejecuta la celda para guardar `identidad.json`.

In [ ]:
ELEGIDO = "gemma4:e2b"  # cambiar por el modelo con la identidad más coherente

final = {
    "identidad": resultados[ELEGIDO],
    "modelo_ollama": ELEGIDO,
    "opciones": OPCIONES,
    "fecha": time.strftime("%Y-%m-%d %H:%M:%S"),
}
with open("identidad.json", "w", encoding="utf-8") as f:
    json.dump(final, f, indent=2, ensure_ascii=False)
print(json.dumps(final, indent=2, ensure_ascii=False))